# 04 - Hybrid Production Architecture

## Scenario: Deterministic Code + LLM Fallbacks

Never use an LLM for something that can be solved with a simple `if` statement. LLMs are slow, expensive, and non-deterministic.

A **Hybrid Architecture** uses traditional API Gateways and deterministic state machines for 90% of requests, and only routes to the LLM Agent for complex, unstructured out-of-band investigations.

In this notebook, we'll build a mock API router for Northstar Support.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

# Attempt to use real API key
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    # Add repo root to path for local execution without pip install
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Deterministic API Gateway

We define a fast, traditional function that handles structured intents (like a button click on a UI).

In [2]:
def deterministic_api(intent: str, user_id: str) -> str:
    """Handles highly structured, predictable requests instantly."""
    if intent == "GET_BALANCE":
        # Fast DB lookup
        return '{"balance": "$150.00"}'
    elif intent == "PASSWORD_RESET":
        # Standard auth flow
        return '{"status": "reset_email_sent"}'
    else:
        return "UNKNOWN_INTENT"


## 2. The LLM Fallback Agent

If the user types a complex, multi-step problem into a chat box, the deterministic API fails. We then catch that failure and invoke the LLM Agent.

In [3]:
def llm_investigation_agent(query: str, user_id: str) -> str:
    print(f"🧠 [Agent] Engaging LLM for unstructured query...")
    # In reality, this would use `client.chat.completions.create(...)`
    # Here we just mock the agent's complex reasoning process
    print("  ...Agent checking logs")
    print("  ...Agent analyzing sentiment")
    return "I found that your last payment failed due to a regional bank outage. I have extended your grace period."

def unified_support_router(query: str, intent: str, user_id: str):
    print(f"\n➡️ Incoming Request from {user_id}: '{query}'")
    
    # Step 1: Try the fast, cheap deterministic path
    result = deterministic_api(intent, user_id)
    
    # Step 2: Fallback to the slow, expensive LLM Agent if needed
    if result == "UNKNOWN_INTENT":
        print("⚠️ Deterministic API failed. Escalating to LLM Agent.")
        result = llm_investigation_agent(query, user_id)
        
    print(f"✅ Final Output: {result}")

# 1. A simple button click (Fast)
unified_support_router("What is my balance?", intent="GET_BALANCE", user_id="cust_111")

# 2. A complex angry email (Slow)
unified_support_router("I tried to pay but it crashed and now I'm locked out!", intent="UNKNOWN", user_id="cust_222")



➡️ Incoming Request from cust_111: 'What is my balance?'
✅ Final Output: {"balance": "$150.00"}

➡️ Incoming Request from cust_222: 'I tried to pay but it crashed and now I'm locked out!'
⚠️ Deterministic API failed. Escalating to LLM Agent.
🧠 [Agent] Engaging LLM for unstructured query...
  ...Agent checking logs
  ...Agent analyzing sentiment
✅ Final Output: I found that your last payment failed due to a regional bank outage. I have extended your grace period.


## Checkpoint

**1. What is the primary benefit of a Hybrid Architecture?**
- A) It uses multiple LLMs at the same time.
- B) It maximizes speed, reliability, and cost-efficiency by reserving the LLM only for tasks that traditional code cannot handle.
- C) It allows the LLM to write its own Python code.
- D) It prevents prompt injections entirely.
